# Route Resilience — Phase 1: Data Pipeline Demo

This notebook runs the **complete Phase 1 data collection and preprocessing pipeline** end-to-end using real satellite datasets.

## Prerequisites

Before running this notebook:

1. Install dependencies:  `pip install -r requirements.txt`
2. Download at least one dataset (see `src/download_utils.py` or README)
3. Ensure `configs/config.yaml` points to your data directories

## Pipeline Steps

| Step | Module | Output |
|------|--------|--------|
| 1 | `ingestion.py` | `dataset_report.json` |
| 2 | `standardization.py` | RGB images + geo metadata |
| 3 | `tiling.py` | 512×512 tiles + `tile_statistics.json` |
| 4 | `splitting.py` | `train/val/test.csv` |
| 5 | `augmentation.py` | Pipeline config + preview PNGs |
| 6 | `occlusion.py` | Occlusion samples + statistics |
| 7 | `quality.py` | `quality_report.json` + `master_dataset.csv` |
| 8 | `visualization.py` | Pipeline summary PNGs |


In [ ]:
import sys
import json
from pathlib import Path

# Ensure project root is in path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0e1117'
matplotlib.rcParams['axes.facecolor'] = '#0e1117'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = '#aaa'
matplotlib.rcParams['xtick.color'] = '#aaa'
matplotlib.rcParams['ytick.color'] = '#aaa'

from src.utils import load_config, get_logger

logger = get_logger('notebook')

# Load config
config = load_config(project_root / 'configs/config.yaml')
print(f"Project: {config['project']['name']}  Phase: {config['project']['phase']}")
print(f"Random seed: {config['project']['random_seed']}")
print(f"Tile size: {config['tiling']['tile_size']}px  Stride: {config['tiling']['stride']}px")

## Step 1: Data Ingestion

In [ ]:
from src.ingestion import run_ingestion

records, dataset_report = run_ingestion(config)

if not records:
    print("\n⚠ No datasets found. Check setup instructions above.")
    print("Run: python src/download_utils.py --dataset check")
else:
    print(f"\n✓ Found {len(records)} image records")
    print(f"  With masks: {sum(1 for r in records if r.has_mask)}")
    print(f"  Georeferenced: {sum(1 for r in records if r.is_georeferenced)}")
    print(f"  Average road %: {dataset_report['class_distribution']['road_pixel_pct_mean']:.2f}%")

    # Display per-dataset breakdown
    df_stats = pd.DataFrame([
        {
            'Dataset': src,
            'Images': stats['total_images'],
            'Masks': stats['total_masks'],
            'Missing Masks': stats['missing_masks'],
            'Avg Road %': f"{stats['road_pixel_pct']['mean']:.2f}%",
            'Georef': stats['georeferenced_count'],
            'Size': stats['total_size'],
        }
        for src, stats in dataset_report['per_dataset_stats'].items()
    ])
    display(df_stats)

In [ ]:
# Show a sample record with geospatial metadata
if records:
    rec = next((r for r in records if r.is_georeferenced), records[0])
    print(f"Sample record: {rec.image_id}")
    print(f"  Source:          {rec.source_dataset}")
    print(f"  Dimensions:      {rec.image_width} × {rec.image_height} ({rec.image_channels} bands)")
    print(f"  dtype:           {rec.image_dtype}")
    print(f"  Road pixels:     {rec.road_pixel_pct:.2f}%")
    print(f"  Georeferenced:   {rec.is_georeferenced}")
    if rec.crs:
        print(f"  CRS:             {rec.crs}")
        print(f"  Bounds:          {rec.bounds}")
        print(f"  Resolution:      {rec.resolution_x:.4f} × {rec.resolution_y:.4f}")
    print(f"  Validation:      {'✓ valid' if rec.is_valid() else '✗ ' + str(rec.validation_errors)}")

## Step 2: Image Standardization

In [ ]:
from src.standardization import run_standardization

if records:
    processed = run_standardization(config, records)
    print(f"✓ Standardized {len(processed)} images")
    
    # Show comparison grid (saved to file)
    comp_path = Path('outputs/visualizations/standardization_comparison.png')
    if comp_path.exists():
        from IPython.display import Image, display
        display(Image(str(comp_path)))
        print(f"Comparison saved at: {comp_path}")

## Step 3: Image Tiling

In [ ]:
from src.tiling import run_tiling

if records:
    tile_infos = run_tiling(config, records)
    
    print(f"\n✓ Kept {len(tile_infos)} tiles")
    
    # Load and display tile stats
    tile_stats_path = Path('outputs/reports/tile_statistics.json')
    if tile_stats_path.exists():
        with open(tile_stats_path) as f:
            tile_stats = json.load(f)
        print(f"  Total generated:  {tile_stats['total_tiles_generated']}")
        print(f"  Kept:             {tile_stats['tiles_kept']}")
        print(f"  Discarded:        {tile_stats['tiles_discarded']}")
        print(f"  Keep rate:        {tile_stats['keep_rate_pct']:.1f}%")
        print(f"  Overlap:          {tile_stats['overlap_pct']:.0f}%")
        print(f"  Discard reasons:  {tile_stats['discard_reasons']}")
    
    # Show tile distribution
    if tile_infos:
        road_pcts = [t.road_pixel_pct for t in tile_infos]
        fig, ax = plt.subplots(1, 1, figsize=(10, 4))
        fig.patch.set_facecolor('#0e1117')
        ax.set_facecolor('#0e1117')
        ax.hist(road_pcts, bins=40, color='#7b68ee', edgecolor='#333')
        ax.set_title('Road Pixel % Distribution in Kept Tiles', color='white')
        ax.set_xlabel('Road Pixel %', color='#aaa')
        ax.set_ylabel('Count', color='#aaa')
        plt.tight_layout()
        plt.show()

## Step 4: Dataset Splitting

In [ ]:
from src.splitting import run_splitting

if tile_infos:
    train_tiles, val_tiles, test_tiles = run_splitting(config, tile_infos)
    
    print(f"✓ Split complete")
    print(f"  Train: {len(train_tiles)} tiles ({len(train_tiles)/len(tile_infos)*100:.1f}%)")
    print(f"  Val:   {len(val_tiles)} tiles ({len(val_tiles)/len(tile_infos)*100:.1f}%)")
    print(f"  Test:  {len(test_tiles)} tiles ({len(test_tiles)/len(tile_infos)*100:.1f}%)")
    
    # Load CSVs
    for split in ['train', 'val', 'test']:
        csv_path = Path(f'outputs/reports/{split}.csv')
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f"  {split}.csv: {len(df)} rows, avg road={df['road_pixel_pct'].mean():.2f}%")
    
    # Source distribution chart
    from collections import Counter
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('#0e1117')
    for ax, (tiles, name) in zip(axes, [(train_tiles, 'Train'), (val_tiles, 'Val'), (test_tiles, 'Test')]):
        sources = Counter(t.source_dataset for t in tiles)
        ax.set_facecolor('#0e1117')
        if sources:
            ax.pie(list(sources.values()), labels=list(sources.keys()),
                   autopct='%1.0f%%', textprops={'color': 'white'})
        ax.set_title(f'{name} ({len(tiles)} tiles)', color='white')
    plt.tight_layout()
    plt.show()

## Step 5: Data Augmentation

In [ ]:
from src.augmentation import run_augmentation
from src.utils import load_image, load_mask

if train_tiles:
    aug_pipeline = run_augmentation(config, train_tiles)
    print(f"✓ Augmentation pipeline built")
    print(f"  Transforms: {len(aug_pipeline.transform.transforms)}")
    
    # Show a preview for the first tile
    tile = train_tiles[0]
    img = load_image(Path(tile.image_tile_path))
    mask = load_mask(Path(tile.mask_tile_path)) if tile.mask_tile_path else np.zeros(img.shape[:2], dtype=np.uint8)
    aug_pipeline.generate_preview(img, mask, image_id='notebook_demo')
    
    preview_path = Path('outputs/visualizations/augmentation_preview_notebook_demo.png')
    if preview_path.exists():
        from IPython.display import Image, display
        display(Image(str(preview_path)))

## Step 6: Occlusion Simulation

In [ ]:
from src.occlusion import run_occlusion

all_tiles = train_tiles or tile_infos
if all_tiles:
    occlusion_results = run_occlusion(config, all_tiles)
    
    print(f"✓ Occlusion simulation complete: {len(occlusion_results)} samples")
    
    if occlusion_results:
        df_occ = pd.DataFrame([
            {
                'Type': r.occlusion_type,
                'Severity': r.severity,
                'Image Coverage %': r.coverage_pct,
                'Road Coverage %': r.road_coverage_pct,
            }
            for r in occlusion_results
        ])
        display(df_occ.groupby(['Type', 'Severity']).mean().round(2))
    
    # Show one comparison
    if occlusion_results:
        result = occlusion_results[0]
        print(f"\nSample occlusion: {result.occlusion_type} — {result.severity}")
        from IPython.display import Image, display
        display(Image(result.comparison_path, width=900))

## Step 7: Quality Analysis

In [ ]:
from src.quality import run_quality_analysis

if records and tile_infos:
    quality_report = run_quality_analysis(
        config, records, tile_infos,
        train_tiles or tile_infos, val_tiles or [], test_tiles or [],
        occlusion_results or []
    )
    
    print("✓ Quality report generated")
    print(f"  Road pixel %:        {quality_report['road_pixel_percentage']['mean']:.2f}% ± {quality_report['road_pixel_percentage']['std']:.2f}%")
    print(f"  Background pixel %:  {quality_report['background_pixel_percentage']['mean']:.2f}%")
    print(f"  Imbalance ratio:     {quality_report['class_imbalance']['imbalance_ratio_bg_to_road']:.1f}:1")
    
    rw = quality_report['road_width_statistics']['avg_road_width_px']
    if rw['mean']:
        print(f"  Avg road width:      {rw['mean']:.1f}px (±{rw['std']:.1f})")
    
    print(f"  Recommendation: {quality_report['class_imbalance']['recommendation']}")
    
    # Show master dataset
    master_path = Path('outputs/reports/master_dataset.csv')
    if master_path.exists():
        df_master = pd.read_csv(master_path)
        print(f"\n  master_dataset.csv: {len(df_master)} rows")
        display(df_master.head())
    
    # Show quality chart
    chart_path = Path('outputs/visualizations/quality_analysis.png')
    if chart_path.exists():
        from IPython.display import Image, display
        display(Image(str(chart_path)))

## Step 8: Pipeline Visualization Summary

In [ ]:
from src.visualization import PipelineVisualizer
from src.utils import load_image, load_mask

if records and tile_infos:
    viz = PipelineVisualizer(config)
    
    # Generate pipeline summary for first valid record
    rec = next((r for r in records if r.has_mask), records[0])
    tile = tile_infos[0]
    
    # Get augmented version
    aug_img, aug_mask = None, None
    if 'aug_pipeline' in dir() and aug_pipeline:
        try:
            img = load_image(Path(tile.image_tile_path))
            mask = load_mask(Path(tile.mask_tile_path)) if tile.mask_tile_path else np.zeros(img.shape[:2], dtype=np.uint8)
            aug_img, aug_mask = aug_pipeline.augment(img.copy(), mask.copy())
        except Exception:
            pass
    
    # Get occluded version
    occ_img = None
    if 'occlusion_results' in dir() and occlusion_results:
        try:
            occ_img = load_image(Path(occlusion_results[0].occluded_path))
        except Exception:
            pass
    
    summary_path = viz.plot_pipeline_summary(rec, tile, aug_img, aug_mask, occ_img)
    print(f"✓ Pipeline summary saved: {summary_path}")
    
    from IPython.display import Image, display
    display(Image(str(summary_path)))

## Phase 1 Complete — Summary

In [ ]:
print("\n" + "="*60)
print("  PHASE 1 PIPELINE COMPLETE")
print("="*60)

output_files = [
    ('outputs/reports/dataset_report.json', 'Dataset Report'),
    ('outputs/reports/quality_report.json', 'Quality Report'),
    ('outputs/reports/master_dataset.csv', 'Master Dataset (source of truth)'),
    ('outputs/reports/train.csv', 'Train Split'),
    ('outputs/reports/val.csv', 'Val Split'),
    ('outputs/reports/test.csv', 'Test Split'),
    ('outputs/reports/tile_statistics.json', 'Tile Statistics'),
    ('outputs/reports/augmentation_pipeline.json', 'Augmentation Config'),
    ('outputs/reports/occlusion_statistics.json', 'Occlusion Statistics'),
    ('outputs/reports/split_metadata.json', 'Split Metadata'),
]

for fpath, label in output_files:
    p = Path(fpath)
    icon = '✓' if p.exists() else '✗'
    size = f" ({p.stat().st_size // 1024} KB)" if p.exists() else ' (missing)'
    print(f"  {icon} {label:40s} {size}")

viz_dir = Path('outputs/visualizations')
if viz_dir.exists():
    viz_files = list(viz_dir.glob('*.png'))
    print(f"\n  Visualizations: {len(viz_files)} PNG files in outputs/visualizations/")

occ_dir = Path('outputs/occlusion_samples')
if occ_dir.exists():
    occ_files = list(occ_dir.glob('*.png'))
    print(f"  Occlusion samples: {len(occ_files)} PNG files in outputs/occlusion_samples/")

print("\n  ✅ Ready for Phase 2: Road Segmentation Model Training")
print("="*60)